# 12 - Plan gold aggregate partition refreshes

Returns the union of capture, observation, queue, claim, and completion dates affected during a configurable lookback window. The gold-refresh pipeline iterates these dates and rebuilds all three date partitions idempotently.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
LOOKBACK_HOURS = 48
DATABASE = ""
TABLE_PREFIX = "people_counter"

In [ ]:
from datetime import datetime, timedelta, timezone
import json
import re

import notebookutils
from pyspark.sql import SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
lookback_hours = int(LOOKBACK_HOURS)
if lookback_hours < 1:
    raise ValueError("LOOKBACK_HOURS must be at least 1")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
planned_at = datetime.now(timezone.utc)
cutoff = planned_at - timedelta(hours=lookback_hours)

work = spark_session.table(table("video_work"))
attempts = spark_session.table(table("video_attempts"))
committed_lines = spark_session.table(table("line_counts_committed"))
recent_committed_work = work.where(
    (F.col("status") == "SUCCEEDED")
    & F.col("committed_attempt_id").isNotNull()
    & (F.col("completed_at") >= F.lit(cutoff))
).select("work_id", "committed_attempt_id", "capture_date")

capture_dates = (
    recent_committed_work
    .select(F.col("capture_date").alias("partition_date"))
)
flow_dates = (
    committed_lines.alias("l")
    .join(
        recent_committed_work.alias("w"),
        (F.col("l.work_id") == F.col("w.work_id"))
        & (F.col("l.attempt_id") == F.col("w.committed_attempt_id")),
        "inner",
    )
    .select(F.to_date("observed_at_utc").alias("partition_date"))
)
queued_dates = work.where(F.col("queued_at") >= F.lit(cutoff)).select(
    F.to_date("queued_at").alias("partition_date")
)
claimed_dates = attempts.where(F.col("claimed_at") >= F.lit(cutoff)).select(
    F.to_date("claimed_at").alias("partition_date")
)
completed_dates = attempts.where(
    F.col("completed_at").isNotNull()
    & (F.col("completed_at") >= F.lit(cutoff))
).select(F.to_date("completed_at").alias("partition_date"))

affected_dates = (
    capture_dates.unionByName(flow_dates)
    .unionByName(queued_dates)
    .unionByName(claimed_dates)
    .unionByName(completed_dates)
    .where(F.col("partition_date").isNotNull())
    .distinct()
    .orderBy("partition_date")
)
items = [
    {"partition_date": row.partition_date.isoformat()}
    for row in affected_dates.collect()
]
outcome = {
    "planned_at": planned_at.isoformat(),
    "cutoff": cutoff.isoformat(),
    "lookback_hours": lookback_hours,
    "partition_count": len(items),
    "items": items,
}
print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))